In [5]:
import os
import re
import gc
import json
import html
import logging
import warnings
import argparse
import unicodedata
from pathlib import Path
from typing import List, Dict, Tuple

os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("DATASETS_VERBOSITY", "error")
os.environ.setdefault("BITSANDBYTES_NOWELCOME", "1")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

import sys
import torch
from torch.utils.data import Dataset, DataLoader
import sacrebleu

import bitsandbytes 

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import logging as hf_logging
from peft import PeftModel

import evaluate
HAVE_HF_BLEU = True

from tqdm.auto import tqdm

hf_logging.set_verbosity_error()

DEBUG = os.environ.get("MT_EVAL_DEBUG", "1") != "0"


def dbg(msg):
    if DEBUG:
        print(f"[DEBUG] {msg}", flush=True)


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--base_model", type=str, default="tencent/Hy-MT2-1.8B")
    p.add_argument("--adapter_dir", type=str, default="wmt26-hymt2")
    p.add_argument("--data_root", type=str, default="tvsub")
    p.add_argument("--src_lang", type=str, default="Chinese")
    p.add_argument("--tgt_lang", type=str, default="English")
    p.add_argument("--num_pairs", type=int, default=200)
    p.add_argument("--batch_size", type=int, default=8)
    p.add_argument("--max_new_tokens", type=int, default=160)
    p.add_argument("--num_beams", type=int, default=4)
    p.add_argument("--load_in_4bit", action="store_true")
    p.add_argument("--out_file", type=str, default="test_hypotheses.txt")

    if argv is None:
        in_notebook = "ipykernel" in sys.modules or "IPython" in sys.modules
        argv = [] if in_notebook else sys.argv[1:]

    args, _unknown = p.parse_known_args(argv)
    return args


_HTML_UNESC = re.compile(r"&(?:amp|apos|quot|lt|gt|#\d+);")
_MULTISPACE = re.compile(r"\s+")
_SPACE_BEFORE_PUNCT = re.compile(r"\s+([,.!?;:%\)\]\}»”’])")
_SPACE_AFTER_PUNCT = re.compile(r"([\(\[\{«“‘])\s+")


def detokenize(text: str) -> str:
    if text is None:
        return ""
    t = text.replace("@-@", "-").replace("@@ ", "").replace(" @@", "")
    t = html.unescape(t)
    t = _HTML_UNESC.sub(lambda m: html.unescape(m.group(0)), t)
    t = _MULTISPACE.sub(" ", t).strip()
    t = _SPACE_BEFORE_PUNCT.sub(r"\1", t)
    t = _SPACE_AFTER_PUNCT.sub(r"\1", t)
    t = unicodedata.normalize("NFKC", t)
    return t


_SEG_RE = re.compile(r'<seg\s+id\s*=\s*"?(\d+)"?\s*>(.*?)</seg>', re.IGNORECASE | re.DOTALL)
_DOC_RE = re.compile(r'<DOC\s+docid\s*=\s*"([^"]+)"\s+sysid\s*=\s*"([^"]+)"[^>]*>(.*?)</DOC>', re.IGNORECASE | re.DOTALL)


def parse_sgm_multiref(path: Path) -> Dict[int, List[str]]:
    raw = Path(path).read_text(encoding="utf-8", errors="ignore")
    docs = _DOC_RE.findall(raw)
    if not docs:
        refs: Dict[int, List[str]] = {}
        for sid, text in _SEG_RE.findall(raw):
            refs.setdefault(int(sid), []).append(detokenize(text))
        return refs

    doc_order: List[str] = []
    doc_local_refs: Dict[str, Dict[int, List[str]]] = {}
    for docid, _sysid, body in docs:
        if docid not in doc_local_refs:
            doc_local_refs[docid] = {}
            doc_order.append(docid)
        for sid, text in _SEG_RE.findall(body):
            doc_local_refs[docid].setdefault(int(sid), []).append(detokenize(text))

    dbg(f"sgm doc_order={doc_order} doc_sizes={[max(doc_local_refs[d].keys()) if doc_local_refs[d] else 0 for d in doc_order]}")

    global_refs: Dict[int, List[str]] = {}
    offset = 0
    for docid in doc_order:
        local = doc_local_refs[docid]
        max_id = max(local.keys()) if local else 0
        for local_id, refs_list in local.items():
            global_refs[offset + local_id] = refs_list
        offset += max_id
    return global_refs


def load_test_pairs(data_root: str, num_pairs: int) -> Tuple[List[str], List[List[str]]]:
    test_src_path = Path(data_root) / "data/processed/test/test_src"
    test_ref_path = Path(data_root) / "data/processed/test/test_trg.sgm"
    dbg(f"test_src_path={test_src_path} exists={test_src_path.exists()}")
    dbg(f"test_ref_path={test_ref_path} exists={test_ref_path.exists()}")
    lines = Path(test_src_path).read_text(encoding="utf-8", errors="ignore").splitlines()
    src_map = {i + 1: detokenize(l) for i, l in enumerate(lines)}
    ref_map = parse_sgm_multiref(test_ref_path)
    dbg(f"parsed src segments={len(src_map)} ref segments={len(ref_map)}")
    ids = sorted(set(src_map.keys()) & set(ref_map.keys()))
    dbg(f"overlapping ids={len(ids)} requested num_pairs={num_pairs}")
    ids = ids[:num_pairs]
    srcs = [src_map[i] for i in ids]
    refs = [ref_map[i] for i in ids]
    dbg(f"final srcs={len(srcs)} refs={len(refs)}")
    if srcs:
        dbg(f"example src[0]={srcs[0]!r}")
        dbg(f"example refs[0]={refs[0]!r}")
    return srcs, refs


PROMPT_TEMPLATE = "Translate the following text into {tgt_lang}. Note that you must ONLY output the translated result without any additional explanation:\n\n{src}"


class MTTestDataset(Dataset):
    def __init__(self, srcs: List[str], tokenizer, tgt_lang: str):
        self.srcs = srcs
        self.tokenizer = tokenizer
        self.tgt_lang = tgt_lang
        self._logged_sample = False

    def __len__(self):
        return len(self.srcs)

    def __getitem__(self, i):
        src = self.srcs[i]
        user_content = PROMPT_TEMPLATE.format(tgt_lang=self.tgt_lang, src=src)
        msgs = [{"role": "user", "content": user_content}]
        prompt_ids = self.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)

        if not self._logged_sample:
            dbg(f"apply_chat_template raw type={type(prompt_ids)}")
            self._logged_sample = True

        if isinstance(prompt_ids, dict):
            prompt_ids = prompt_ids["input_ids"]
        if hasattr(prompt_ids, "input_ids") and not isinstance(prompt_ids, (list, str)):
            prompt_ids = prompt_ids.input_ids
        if isinstance(prompt_ids, str):
            prompt_ids = self.tokenizer(prompt_ids, add_special_tokens=False)["input_ids"]
        if isinstance(prompt_ids, list) and len(prompt_ids) > 0 and isinstance(prompt_ids[0], list):
            prompt_ids = prompt_ids[0]

        if not all(isinstance(t, int) for t in prompt_ids):
            dbg(f"non-int tokens found, coercing: sample={prompt_ids[:10]}")
            prompt_ids = [int(t) for t in prompt_ids]

        return {"input_ids": prompt_ids, "src": src}


def make_collate(tokenizer):
    def collate(batch):
        for j, b in enumerate(batch):
            if not isinstance(b["input_ids"], list) or not all(isinstance(t, int) for t in b["input_ids"]):
                raise TypeError(f"batch item {j} has malformed input_ids: type={type(b['input_ids'])} sample={str(b['input_ids'])[:200]}")
        maxlen = max(len(b["input_ids"]) for b in batch)
        pad = tokenizer.pad_token_id
        ids = torch.full((len(batch), maxlen), pad, dtype=torch.long)
        attn = torch.zeros((len(batch), maxlen), dtype=torch.long)
        for i, b in enumerate(batch):
            L = len(b["input_ids"])
            ids[i, maxlen - L:] = torch.tensor(b["input_ids"], dtype=torch.long)
            attn[i, maxlen - L:] = 1
        srcs = [b["src"] for b in batch]
        return {"input_ids": ids, "attention_mask": attn, "srcs": srcs}
    return collate


def compute_metrics(hyps: List[str], refs_multi: List[List[str]]) -> Dict[str, float]:
    dbg(f"compute_metrics hyps={len(hyps)} refs_multi={len(refs_multi)}")
    out = {}
    max_r = max(len(r) for r in refs_multi) if refs_multi else 1
    dbg(f"max_refs_per_segment={max_r}")
    padded = [(r + [r[-1]] * (max_r - len(r))) if r else [""] * max_r for r in refs_multi]
    ref_lists = list(map(list, zip(*padded))) if padded else [[""] * len(hyps)]

    sb = sacrebleu.corpus_bleu(hyps, ref_lists, tokenize="13a", smooth_method="exp")
    out["sacreBLEU"] = float(sb.score)
    dbg(f"sacreBLEU={out['sacreBLEU']:.4f}")

    chrf = sacrebleu.corpus_chrf(hyps, ref_lists)
    out["chrF"] = float(chrf.score)
    dbg(f"chrF={out['chrF']:.4f}")

    try:
        ter = sacrebleu.corpus_ter(hyps, ref_lists)
        out["TER"] = float(ter.score)
        dbg(f"TER={out['TER']:.4f}")
    except Exception as e:
        out["TER"] = float("nan")
        dbg(f"TER computation failed: {e}")

    if HAVE_HF_BLEU:
        try:
            bleu_hf = evaluate.load("bleu")
            b = bleu_hf.compute(predictions=hyps, references=refs_multi)
            out["BLEU"] = float(b["bleu"]) * 100.0
            dbg(f"BLEU={out['BLEU']:.4f}")
        except Exception as e:
            out["BLEU"] = float("nan")
            dbg(f"BLEU computation failed: {e}")
    else:
        out["BLEU"] = float("nan")
        dbg("HF evaluate BLEU unavailable")

    return out


def main(**overrides):
    args = parse_args()
    for k, v in overrides.items():
        setattr(args, k, v)

    dbg(f"args={vars(args)}")
    dbg(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        dbg(f"cuda_version={torch.version.cuda} device_name={torch.cuda.get_device_name(0)}")
    dbg(f"bitsandbytes_loaded={getattr(bitsandbytes, '__file__', None) is not None}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if bf16_ok else torch.float16
    dbg(f"device={device} bf16_ok={bf16_ok} compute_dtype={compute_dtype}")

    dbg("loading tokenizer")
    tokenizer = AutoTokenizer.from_pretrained(args.base_model, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    dbg(f"tokenizer loaded pad_token_id={tokenizer.pad_token_id} eos_token_id={tokenizer.eos_token_id}")

    if args.load_in_4bit:
        if not getattr(bitsandbytes, "__file__", None):
            raise RuntimeError(
                "bitsandbytes failed to load its native CUDA library, so --load_in_4bit "
                "is unavailable. Run `pip uninstall bitsandbytes -y` and either reinstall a "
                "version matching your CUDA/driver, or drop --load_in_4bit (a 1.8B model "
                "runs fine in fp16/bf16)."
            )
        dbg("loading base model in 4bit")
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype,
        )
        base_model = AutoModelForCausalLM.from_pretrained(
                args.base_model,
                quantization_config=bnb_cfg,
                device_map={"": 0} if torch.cuda.is_available() else "auto",
                trust_remote_code=True,
                torch_dtype=compute_dtype,
                attn_implementation="sdpa",)
    else:
        dbg("loading base model without quantization")
        base_model = AutoModelForCausalLM.from_pretrained(
                args.base_model,
                device_map={"": 0} if torch.cuda.is_available() else "auto",
                trust_remote_code=True,
                torch_dtype=compute_dtype,
                attn_implementation="sdpa",)
    base_model.config.use_cache = True
    dbg(f"base model loaded device={next(base_model.parameters()).device} dtype={next(base_model.parameters()).dtype}")

    dbg(f"loading adapter from {args.adapter_dir}")
    model = PeftModel.from_pretrained(base_model, args.adapter_dir)
    model.eval()
    dbg("adapter loaded and model set to eval mode")

    srcs, refs_multi = load_test_pairs(args.data_root, args.num_pairs)
    dbg(f"loaded {len(srcs)} test pairs")

    ds = MTTestDataset(srcs, tokenizer, args.tgt_lang)
    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=False,
                         collate_fn=make_collate(tokenizer), num_workers=0, pin_memory=True)
    dbg(f"dataloader built batches={len(loader)} batch_size={args.batch_size}")

    hyps = []
    dbg("entering generation loop")
    with torch.no_grad():
        for bi, batch in enumerate(tqdm(loader, desc="generating")):
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            dbg(f"batch {bi} input_ids_shape={tuple(input_ids.shape)}")
            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attn,
                max_new_tokens=args.max_new_tokens,
                num_beams=args.num_beams,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
            new_tokens = gen[:, input_ids.shape[1]:]
            for row in new_tokens:
                txt = tokenizer.decode(row, skip_special_tokens=True).strip()
                hyps.append(txt)
            if bi == 0:
                dbg(f"sample hypothesis[0]={hyps[0]!r}")

    dbg(f"generation complete total_hyps={len(hyps)}")
    metrics = compute_metrics(hyps, refs_multi)

    print("\nTest set size:", len(hyps))
    for k in ["BLEU", "sacreBLEU", "chrF", "TER"]:
        print(f"{k}: {metrics.get(k, float('nan')):.4f}")

    out_path = Path(args.out_file)
    out_path.write_text("\n".join(hyps), encoding="utf-8")
    dbg(f"wrote hypotheses to {out_path.resolve()}")

    metrics_path = out_path.with_suffix(".metrics.json")
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    dbg(f"wrote metrics to {metrics_path.resolve()}")

    del model, base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    dbg("cleanup complete")


if __name__ == "__main__":
    main(load_in_4bit=True)

[DEBUG] args={'base_model': 'tencent/Hy-MT2-1.8B', 'adapter_dir': 'wmt26-hymt2', 'data_root': 'tvsub', 'src_lang': 'Chinese', 'tgt_lang': 'English', 'num_pairs': 200, 'batch_size': 8, 'max_new_tokens': 160, 'num_beams': 4, 'load_in_4bit': True, 'out_file': 'test_hypotheses.txt'}
[DEBUG] torch=2.13.0+cu130 cuda_available=True
[DEBUG] cuda_version=13.0 device_name=NVIDIA GeForce RTX 4070 Laptop GPU
[DEBUG] bitsandbytes_loaded=True
[DEBUG] device=cuda bf16_ok=True compute_dtype=torch.bfloat16
[DEBUG] loading tokenizer


[DEBUG] tokenizer loaded pad_token_id=120002 eos_token_id=120020
[DEBUG] loading base model in 4bit


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

[DEBUG] base model loaded device=cuda:0 dtype=torch.bfloat16
[DEBUG] loading adapter from wmt26-hymt2
[DEBUG] adapter loaded and model set to eval mode
[DEBUG] test_src_path=tvsub\data\processed\test\test_src exists=True
[DEBUG] test_ref_path=tvsub\data\processed\test\test_trg.sgm exists=True
[DEBUG] sgm doc_order=['SUB01', 'SUB02'] doc_sizes=[330, 824]
[DEBUG] parsed src segments=1154 ref segments=1154
[DEBUG] overlapping ids=1154 requested num_pairs=200
[DEBUG] final srcs=200 refs=200
[DEBUG] example src[0]='手续 办得 还 顺利 吗?'
[DEBUG] example refs[0]=['did everything go okay with the annulment?', "how 'd it go?", 'how did it go?']
[DEBUG] loaded 200 test pairs
[DEBUG] dataloader built batches=25 batch_size=8
[DEBUG] entering generation loop


generating:   0%|          | 0/25 [00:00<?, ?it/s]

[DEBUG] apply_chat_template raw type=<class 'transformers.tokenization_utils_base.BatchEncoding'>
[DEBUG] batch 0 input_ids_shape=(8, 38)
[DEBUG] sample hypothesis[0]="how 's the paperwork going?"
[DEBUG] batch 1 input_ids_shape=(8, 42)
[DEBUG] batch 2 input_ids_shape=(8, 49)
[DEBUG] batch 3 input_ids_shape=(8, 51)
[DEBUG] batch 4 input_ids_shape=(8, 48)
[DEBUG] batch 5 input_ids_shape=(8, 55)
[DEBUG] batch 6 input_ids_shape=(8, 54)
[DEBUG] batch 7 input_ids_shape=(8, 50)
[DEBUG] batch 8 input_ids_shape=(8, 54)
[DEBUG] batch 9 input_ids_shape=(8, 47)
[DEBUG] batch 10 input_ids_shape=(8, 50)
[DEBUG] batch 11 input_ids_shape=(8, 54)
[DEBUG] batch 12 input_ids_shape=(8, 50)
[DEBUG] batch 13 input_ids_shape=(8, 51)
[DEBUG] batch 14 input_ids_shape=(8, 51)
[DEBUG] batch 15 input_ids_shape=(8, 50)
[DEBUG] batch 16 input_ids_shape=(8, 48)
[DEBUG] batch 17 input_ids_shape=(8, 53)
[DEBUG] batch 18 input_ids_shape=(8, 55)
[DEBUG] batch 19 input_ids_shape=(8, 46)
[DEBUG] batch 20 input_ids_shape=